# Earnings Call Analyzer — Project Walkthrough

**Author:** Chamundeswari Murugavel  
**Repo:** [`earnings-call-analyzer`](../README.md)  
**One-liner:** An NLP pipeline that turns earnings-call transcripts into a directional T+1 return classifier, then backtests the signal against realised excess returns over SPY — served as a FastAPI service with a Streamlit UI.

## The thesis

> The *headline EPS number* is priced into the stock within milliseconds. The *language management uses* — how much they hedge, whether their forward guidance turned more cautious, how their tone shifted vs last quarter — takes longer to diffuse.

This is not a novel claim. It has a long academic lineage:

- **Loughran & McDonald (2011)** — built the finance-specific sentiment dictionary that almost every modern paper uses. Showed that generic sentiment lexicons (e.g. Harvard IV-4) systematically mis-classify finance words (`liability`, `tax`, `vice`) and that an uncertainty / weak-modal word list predicts post-event drift.
- **Hassan, Hollander, van Lent & Tahoun (2019, *QJE*)** — quantified *political risk* from earnings calls and showed it predicts firm-level investment and hiring decisions.
- **Larcker & Zakolyukina (2012)** — showed that deceptive calls have measurable linguistic signatures (more references to general knowledge, fewer non-extreme positive emotion words).

What this project does is turn that literature into a reproducible, testable pipeline that a recruiter can `git clone` and `pip install -e .` in under five minutes.

## What's in this notebook

1. **Architecture** — the data flow end-to-end
2. **Data sourcing & a note on data ethics** — why I do *not* scrape Seeking Alpha
3. **Feature engineering** — FinBERT, hedge words, forward-looking statements, QoQ tone shift
4. **Labelling** — T+1 excess return vs SPY (why excess, not raw)
5. **Model** — LightGBM under walk-forward cross-validation (why not random K-fold)
6. **Backtest** — long/short rule, hit-rate, Sharpe, drawdown vs SPY buy-and-hold
7. **Productionisation** — FastAPI + Streamlit
8. **Honest limitations & next steps**

> **How to read this notebook:** every section runs on a small *synthetic* dataset so the whole notebook executes in ~30 seconds with no external downloads, no GPU, and no API keys. The production paths (FinBERT, EDGAR, yfinance, MLflow) are imported and demonstrated but switched off via flags. The README explains how to flip the flags and run the real pipeline.


## 1. Architecture

```
┌─────────────────┐    ┌─────────────────┐
│  SEC EDGAR 8-K  │    │ HF earnings_call│        (licensed / public sources only)
│   Item 2.02     │    │     corpus      │
└────────┬────────┘    └────────┬────────┘
         │                      │
         └──────────┬───────────┘
                    ▼
         ┌──────────────────────┐
         │ eca.features.build   │   FinBERT sentiment (sentence-level)
         │                      │   + hedge-word ratio (Loughran-McDonald)
         │                      │   + forward-looking statement detector
         │                      │   + QoQ tone-shift deltas
         └──────────┬───────────┘
                    ▼
         ┌──────────────────────┐    yfinance close-to-close,
         │ eca.prices.attach_…  │    label = sign(ret_T+1 − ret_SPY)
         └──────────┬───────────┘
                    ▼
         ┌──────────────────────┐
         │ eca.model.train      │    LightGBM, walk-forward CV, MLflow
         └──────────┬───────────┘
                    ▼
         ┌──────────────────────┐
         │ eca.backtest.engine  │    long if p≥0.5+τ, short if p≤0.5-τ
         └──────────┬───────────┘
                    ▼
         ┌──────────────────────┐
         │ eca.api  +  streamlit│    POST /analyze, GET /backtest/{ticker}
         └──────────────────────┘
```

Every box maps to a real Python module in `src/eca/`. The boundaries are deliberate: each layer can be unit-tested in isolation and swapped (e.g. FinBERT → DeBERTa-v3-finance) without touching the rest.


## 2. Data sourcing & a note on data ethics

The first temptation when building a project like this is to point a scraper at Seeking Alpha or Motley Fool and call it a day. I deliberately did **not** do that, and I think the choice itself is part of the portfolio story:

| Source | Status in this repo | Why |
| --- | --- | --- |
| **SEC EDGAR 8-K (Item 2.02)** | ✅ primary loader (`eca.ingest.edgar`) | Public, official, no ToS issue. Polite-rate-limited per SEC's 10 req/s rule. |
| **HuggingFace `jlh-ibm/earnings_call`** | ✅ bulk-training loader (`eca.ingest.hf_dataset`) | Licensed academic corpus, ~3.5k transcripts. Streamed (no full download). |
| **Motley Fool** | ⚠️ opt-in only (`eca.ingest.motley_fool`) | ToS prohibits scraping. The fetcher honours `robots.txt` and rate-limits to 1 req / 2 s. Use only for personal research. |
| **Seeking Alpha** | ❌ deliberately not implemented | Hard paywall + aggressive bot detection. Scraping it would breach ToS and likely the CFAA. |

A quant researcher who respects data licensing is a quant researcher who won't get their employer sued. That's worth being explicit about.


## 3. Setup

We import from the package, fix a seed, and build a small **synthetic corpus** so the notebook is fully reproducible offline. Each fake transcript is generated from a per-firm latent "tone" parameter that biases the proportion of bullish vs cautious phrasing. The model should learn to recover that tone — if it can't even do that on data with a known signal-to-noise ratio, the production pipeline has no hope.


In [ ]:
from __future__ import annotations

import sys
from datetime import date, timedelta
from pathlib import Path

import numpy as np
import pandas as pd

# make `src/` importable when running the notebook from the repo root
REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO_ROOT / "src"))

from eca.features.guidance import guidance_features
from eca.features.hedging import hedge_features
from eca.features.tone_shift import add_qoq_tone_shifts
from eca.ingest.schema import Transcript

RNG = np.random.default_rng(7)
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 160)

print(f"repo root: {REPO_ROOT}")
print(f"numpy {np.__version__}  |  pandas {pd.__version__}")


In [ ]:
"""Synthetic transcript generator.

Each ticker has a latent ``tone`` in [-1, 1]. When we generate a quarter we add
Gaussian drift to the tone, then sample sentences from bullish / cautious /
hedging templates with probabilities driven by that tone. The *truth* we want
the model to recover is: high-tone calls drift positively on T+1, low-tone
calls drift negatively (with noise).
"""

BULLISH = [
    "We expect record revenue next quarter driven by strong demand across all segments.",
    "Our outlook for the full year remains very robust and we are raising guidance.",
    "We anticipate continued double-digit growth in our core platform.",
    "Margin expansion accelerated this quarter and we see further tailwinds ahead.",
    "Customer adoption is exceeding our internal targets going forward.",
    "We are seeing strong momentum entering Q4 and beyond.",
]
CAUTIOUS = [
    "We expect some softness in the next quarter due to macro headwinds.",
    "Our guidance for the full year reflects a more challenging environment.",
    "We anticipate decline in certain end markets going forward.",
    "Margin pressure is likely to weigh on results in the coming quarters.",
    "We see uncertainty around customer spending patterns in the second half.",
    "Weakness in enterprise demand may continue into next year.",
]
HEDGE = [
    "Results could vary depending on macroeconomic conditions.",
    "We believe outcomes may fluctuate based on FX assumptions.",
    "It is possible that approximately 5 percent of orders slip into Q1.",
    "Estimates suggest tentative recovery but visibility remains limited.",
    "Predictions around demand remain uncertain and could shift.",
]
NEUTRAL = [
    "Operating expenses came in line with our internal plan this quarter.",
    "Headcount grew modestly as we continued hiring in R&D.",
    "Cash and equivalents stood at approximately twelve billion dollars at quarter end.",
    "We returned capital to shareholders via buybacks and dividends as planned.",
    "Inventory levels remained stable across the major product categories.",
]
QA_TEMPLATE = (
    "Question and Answer Session.\n"
    "Analyst: Thanks for taking my question. {q1}\n"
    "CEO: {a1}\n"
    "Analyst: A quick follow-up on guidance. {q2}\n"
    "CFO: {a2}\n"
)
Q_TEMPLATES = [
    "Can you talk about pricing dynamics in the channel?",
    "How should we think about gross margin over the next year?",
    "What are you seeing in enterprise demand?",
]


def _make_call_text(tone: float, n_paragraphs: int = 22) -> tuple[str, str]:
    """Return (prepared_remarks, qa_section). Higher tone -> more bullish phrasing."""
    p_bull = float(np.clip(0.5 + 0.5 * tone, 0.05, 0.95))
    p_cau = 1 - p_bull
    sentences: list[str] = ["Good afternoon and welcome to our earnings call."]
    for _ in range(n_paragraphs):
        roll = RNG.random()
        if roll < 0.55:
            pool = BULLISH if RNG.random() < p_bull else CAUTIOUS
        elif roll < 0.80:
            pool = NEUTRAL
        else:
            pool = HEDGE
        sentences.append(str(RNG.choice(pool)))
    prepared = " ".join(sentences)
    qa = QA_TEMPLATE.format(
        q1=RNG.choice(Q_TEMPLATES),
        a1=RNG.choice(BULLISH if RNG.random() < p_bull else CAUTIOUS),
        q2=RNG.choice(Q_TEMPLATES),
        a2=RNG.choice(BULLISH if RNG.random() < p_bull else HEDGE),
    )
    return prepared, qa


def make_corpus(n_tickers: int = 12, n_quarters: int = 12, start: date = date(2022, 1, 15)) -> list[Transcript]:
    tickers = [f"SYN{i:02d}" for i in range(n_tickers)]
    base_tones = RNG.uniform(-0.6, 0.6, size=n_tickers)
    rows: list[Transcript] = []
    for ti, ticker in enumerate(tickers):
        tone = float(base_tones[ti])
        for q in range(n_quarters):
            tone = float(np.clip(tone + RNG.normal(0, 0.25), -1, 1))  # drift
            prepared, qa = _make_call_text(tone)
            call_date = start + timedelta(days=91 * q + int(RNG.integers(-3, 4)))
            rows.append(
                Transcript(
                    ticker=ticker,
                    call_date=call_date,
                    fiscal_quarter=f"Q{(q % 4) + 1} {call_date.year}",
                    prepared_remarks=prepared,
                    qa_section=qa,
                    source="synthetic",
                    metadata={"latent_tone": tone},
                )
            )
    return rows


corpus = make_corpus()
print(f"generated {len(corpus)} synthetic transcripts over {len({t.ticker for t in corpus})} tickers")
print("\nexample (first 280 chars of one transcript):\n")
print(corpus[0].prepared_remarks[:280], "…")


## 4. Feature engineering

Four families of features, each with a published rationale:

| Feature | Module | Rationale |
| --- | --- | --- |
| **FinBERT sentiment** (pos/neu/neg sentence probs, share-positive, polarity mean/std) | `eca.features.sentiment` | FinBERT (Araci 2019) is fine-tuned on financial news; outperforms generic BERT on the Financial PhraseBank by ~10 pts F1. |
| **Hedge-word ratio** (`could`, `may`, `approximately`, …) | `eca.features.hedging` | Loughran-McDonald (2011) uncertainty + weak-modal dictionaries. Higher hedging predicts negative post-event drift. |
| **Forward-looking statements** (sentence contains time-cue ∧ expectation-verb; count, ratio, ±polarity) | `eca.features.guidance` | Hassan et al. (2019) framework. The *direction* of guidance language is what matters, not the level. |
| **QoQ tone shift** (Δ vs prior call) | `eca.features.tone_shift` | The *change* in tone vs the prior quarter is the alpha. A perpetually-bullish CEO whose tone slips is more informative than a one-off cautious quote. |

For this notebook we'll use the real `hedging` and `guidance` modules, and substitute a fast lexicon-based stand-in for FinBERT so the demo doesn't need a 400 MB model download. The production path uses the actual FinBERT — see `eca.features.sentiment.FinBertSentiment`.


In [ ]:
"""Fast lexicon stand-in for FinBERT (notebook-only). Production uses ProsusAI/finbert.

The contract is the same as ``eca.features.sentiment.FinBertSentiment.score``:
return a SentimentAgg with sentence-level statistics. This lets us validate the
downstream pipeline without a 400 MB model download.
"""
import re

from eca.features.sentiment import SentimentAgg

_POS_LEX = {
    "growth", "record", "strong", "robust", "accelerate", "accelerating",
    "expand", "expansion", "improve", "improving", "opportunity", "momentum",
    "tailwind", "beat", "exceed", "outperform", "raising",
}
_NEG_LEX = {
    "decline", "weak", "weakness", "headwind", "headwinds", "soft", "softening",
    "pressure", "challenging", "slow", "slowing", "miss", "below", "disappoint",
    "uncertain", "uncertainty",
}
_SENT_SPLIT = re.compile(r"(?<=[.!?])\s+(?=[A-Z])")
_WORD = re.compile(r"[A-Za-z']+")


def fast_sentiment(text: str) -> SentimentAgg:
    sents = [s for s in _SENT_SPLIT.split(re.sub(r"\s+", " ", text).strip()) if 20 <= len(s) <= 1000]
    if not sents:
        return SentimentAgg(0, 0, 0, 0, 0, 0, 0, 0)
    pos_arr, neg_arr, neu_arr = [], [], []
    for s in sents:
        words = _WORD.findall(s.lower())
        p = sum(1 for w in words if w in _POS_LEX) / max(len(words), 1)
        n = sum(1 for w in words if w in _NEG_LEX) / max(len(words), 1)
        # squash to pseudo-probabilities so the scale matches FinBERT
        p_score = min(1.0, p * 12)
        n_score = min(1.0, n * 12)
        u_score = max(0.0, 1.0 - p_score - n_score)
        pos_arr.append(p_score); neg_arr.append(n_score); neu_arr.append(u_score)
    pos_a, neg_a, neu_a = np.array(pos_arr), np.array(neg_arr), np.array(neu_arr)
    polarity = pos_a - neg_a
    labels = np.argmax(np.stack([pos_a, neg_a, neu_a], axis=1), axis=1)
    return SentimentAgg(
        sent_pos_mean=float(pos_a.mean()),
        sent_neg_mean=float(neg_a.mean()),
        sent_neu_mean=float(neu_a.mean()),
        sent_pos_share=float((labels == 0).mean()),
        sent_neg_share=float((labels == 1).mean()),
        sent_polarity_mean=float(polarity.mean()),
        sent_polarity_std=float(polarity.std()),
        n_sentences=int(len(sents)),
    )


# Smoke-test on one transcript so we know the stand-in is sensible.
demo = fast_sentiment(corpus[0].full_text)
demo.as_dict()


In [ ]:
def build_row(t: Transcript) -> dict:
    """Mirror of eca.features.build.build_features but using the fast sentiment shim."""
    full = t.full_text
    sent = fast_sentiment(full).as_dict()
    hedge = hedge_features(full)
    fls = guidance_features(full)
    prepared_len = len(t.prepared_remarks)
    qa_len = len(t.qa_section)
    qa_ratio = qa_len / (prepared_len + qa_len) if (prepared_len + qa_len) else 0.0
    return {
        "ticker": t.ticker,
        "call_date": t.call_date,
        "fiscal_quarter": t.fiscal_quarter,
        "latent_tone": t.metadata.get("latent_tone"),
        **sent,
        **hedge,
        **fls,
        "prepared_len_chars": prepared_len,
        "qa_len_chars": qa_len,
        "qa_ratio": qa_ratio,
    }


features_df = pd.DataFrame([build_row(t) for t in corpus])
features_df = add_qoq_tone_shifts(features_df)
print(f"shape: {features_df.shape}")
features_df.head()


### Feature distributions

A quick sanity check: do the four feature families look reasonable? We want non-degenerate distributions for the model to learn from, and we want the QoQ-delta columns to be roughly mean-zero (otherwise we have a calendar-effect leak).


In [ ]:
import matplotlib.pyplot as plt

FEATURE_GROUPS = {
    "Sentiment (level)": ["sent_polarity_mean", "sent_pos_share", "sent_neg_share"],
    "Hedging": ["hedge_ratio"],
    "Forward-looking": ["fls_ratio", "fls_positive", "fls_negative"],
    "QoQ tone shift (delta)": ["sent_polarity_mean_dqoq", "hedge_ratio_dqoq", "fls_positive_dqoq"],
}

fig, axes = plt.subplots(2, 2, figsize=(11, 7))
for ax, (title, cols) in zip(axes.flat, FEATURE_GROUPS.items()):
    for c in cols:
        if c in features_df.columns:
            features_df[c].dropna().hist(ax=ax, bins=25, alpha=0.55, label=c)
    ax.set_title(title); ax.legend(fontsize=8); ax.grid(alpha=0.3)
plt.suptitle("Feature distributions on the synthetic corpus", y=1.02)
plt.tight_layout(); plt.show()


## 5. Labelling — T+1 *excess* return

`eca.prices.attach_labels` calls yfinance, computes the close-to-close return from the first trading day after the call to the next, then **subtracts SPY's same-window return** before taking the sign.

Why excess instead of raw? An earnings beat on a day the whole market falls 3 % still implies the call carried information. Raw returns would attribute that market-wide move to the call's language. Excess returns isolate the *idiosyncratic* part.

For the synthetic demo we generate the label directly from the latent tone plus noise — the model's job is to recover the tone-→-label mapping from the engineered features alone (it never sees `latent_tone`).


In [ ]:
# Synthetic excess return: a noisy linear function of latent_tone (and a touch of hedging).
# Real pipeline: eca.prices.attach_labels(features_df) — calls yfinance + SPY benchmark.
noise = RNG.normal(0, 0.02, size=len(features_df))
features_df["ret_excess"] = (
    0.018 * features_df["latent_tone"]
    - 0.010 * features_df["hedge_ratio"].fillna(0)
    + noise
)
features_df["ret_bench"] = RNG.normal(0.0003, 0.011, size=len(features_df))  # SPY daily
features_df["ret_raw"] = features_df["ret_excess"] + features_df["ret_bench"]
features_df["label"] = (features_df["ret_excess"] > 0).astype(int)

print(f"label distribution: {features_df['label'].value_counts().to_dict()}")
print(f"mean |ret_excess|: {features_df['ret_excess'].abs().mean():.4f}")
features_df[["ticker", "call_date", "latent_tone", "ret_excess", "label"]].head()


## 6. Model — why walk-forward, not random K-fold

Earnings-call returns are **time-ordered** and weakly autocorrelated within a regime. A random K-fold of (X, y) leaks future information into the training set — the model sees a 2024 call in training and is tested on a 2023 call. The reported accuracy is then optimistically biased and won't survive paper-trading.

Walk-forward CV (`sklearn.model_selection.TimeSeriesSplit`) trains on `[0..k]` and tests on `[k+1..k+w]` for expanding windows of `k`. This respects causality. The cost is fewer effective training samples in the early folds, which is honest.

The model itself is **LightGBM**. Choice rationale:
- Handles missing values natively (FinBERT can fail on short calls and produce NaNs).
- Captures non-linear interactions (e.g. *high hedge ratio is bad, but only when polarity is also negative*) without manual feature crosses.
- Trains in seconds on the dataset size we have (~3k transcripts), so we can iterate on features rather than wait on a GPU.
- Ships well-tuned defaults; we just nudge `num_leaves` and `min_data_in_leaf` for the small-data regime.

A logistic-regression baseline is included below so reviewers can see the LightGBM lift over a linear model.


In [ ]:
import lightgbm as lgb
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, log_loss, roc_auc_score
from sklearn.model_selection import TimeSeriesSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

FEATURE_COLS = [
    "sent_pos_mean", "sent_neg_mean", "sent_neu_mean",
    "sent_pos_share", "sent_neg_share",
    "sent_polarity_mean", "sent_polarity_std",
    "hedge_count", "hedge_ratio", "n_words",
    "fls_count", "fls_ratio", "fls_positive", "fls_negative",
    "n_sentences",
    "prepared_len_chars", "qa_len_chars", "qa_ratio",
    "sent_pos_mean_dqoq", "sent_neg_mean_dqoq", "sent_polarity_mean_dqoq",
    "hedge_ratio_dqoq", "fls_positive_dqoq", "fls_negative_dqoq", "fls_ratio_dqoq",
]
FEATURE_COLS = [c for c in FEATURE_COLS if c in features_df.columns]

# Sort by time — walk-forward demands it.
data = features_df.sort_values("call_date").reset_index(drop=True)
X = data[FEATURE_COLS].astype(float).fillna(0.0)
y = data["label"].astype(int)

splitter = TimeSeriesSplit(n_splits=5)
records: list[dict] = []

for fold, (tr, te) in enumerate(splitter.split(X), 1):
    Xtr, Xte = X.iloc[tr], X.iloc[te]
    ytr, yte = y.iloc[tr], y.iloc[te]
    if ytr.nunique() < 2 or yte.nunique() < 2:
        continue

    # LightGBM
    lgbm = lgb.LGBMClassifier(
        n_estimators=400, learning_rate=0.05, num_leaves=31,
        min_data_in_leaf=10, feature_fraction=0.9, bagging_fraction=0.9,
        bagging_freq=5, verbose=-1,
    )
    lgbm.fit(Xtr, ytr, eval_set=[(Xte, yte)], callbacks=[lgb.early_stopping(30, verbose=False)])
    p_lgb = lgbm.predict_proba(Xte)[:, 1]

    # Logistic baseline
    logreg = Pipeline([("sc", StandardScaler()), ("lr", LogisticRegression(max_iter=500))])
    logreg.fit(Xtr, ytr)
    p_lr = logreg.predict_proba(Xte)[:, 1]

    for name, p in [("lightgbm", p_lgb), ("logreg", p_lr)]:
        records.append({
            "fold": fold, "model": name,
            "accuracy": accuracy_score(yte, (p >= 0.5).astype(int)),
            "auc": roc_auc_score(yte, p),
            "log_loss": log_loss(yte, np.clip(p, 1e-6, 1 - 1e-6)),
            "n_train": len(tr), "n_test": len(te),
        })

cv_df = pd.DataFrame(records)
cv_summary = cv_df.groupby("model")[["accuracy", "auc", "log_loss"]].mean().round(3)
print("Walk-forward CV results (mean over 5 folds):")
print(cv_summary)


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
for model, sub in cv_df.groupby("model"):
    ax.plot(sub["fold"], sub["accuracy"], marker="o", label=model)
ax.axhline(0.5, ls="--", c="grey", alpha=0.6, label="coin-flip")
ax.set_title("Walk-forward accuracy per fold")
ax.set_xlabel("fold (1 = oldest test window)"); ax.set_ylabel("accuracy")
ax.set_ylim(0.35, 1.0); ax.grid(alpha=0.3); ax.legend()
plt.tight_layout(); plt.show()


### Feature importance — what does the model actually use?

The model should lean on the QoQ deltas and the polarity terms. If it loads on `n_words` or `qa_len_chars` instead, the signal is leaking through transcript length rather than content — and we'd need to dig in.


In [ ]:
# Refit on all data to get a stable importance ranking.
final = lgb.LGBMClassifier(
    n_estimators=400, learning_rate=0.05, num_leaves=31,
    min_data_in_leaf=10, verbose=-1,
).fit(X, y)

imp = pd.Series(final.feature_importances_, index=FEATURE_COLS).sort_values(ascending=True).tail(15)

fig, ax = plt.subplots(figsize=(7, 5))
imp.plot.barh(ax=ax, color="steelblue")
ax.set_title("Top-15 LightGBM feature importances (split count)")
ax.grid(alpha=0.3); plt.tight_layout(); plt.show()


## 7. Backtest — does accuracy translate to PnL?

A classifier with 60 % accuracy is not automatically profitable. The asymmetry between winners and losers, position sizing, and confidence thresholding all dominate. The backtest harness (`eca.backtest.engine.run_backtest`) implements a deliberately simple rule so the result is unambiguous:

- At the close of day T (call day), enter long if `prob_up ≥ 0.5 + τ`, short if `prob_up ≤ 0.5 - τ`, flat otherwise.
- Unwind at the close of day T+1.
- Equal-weight across positions opened on the same day.
- Daily PnL = mean of trade returns.

Reported metrics: `n_trades`, `hit_rate`, `mean_trade_return`, `total_return`, `annualised_sharpe`, `max_drawdown`, and the SPY benchmark over the same calendar.

**To honour walk-forward**, we should use *out-of-fold* probabilities (from the CV) rather than refit-on-everything probabilities. We do that below.


In [ ]:
from eca.backtest import run_backtest

# Re-run walk-forward but this time keep the out-of-fold predictions for honest backtesting.
oof = np.full(len(X), np.nan)
for tr, te in splitter.split(X):
    if y.iloc[tr].nunique() < 2 or y.iloc[te].nunique() < 2:
        continue
    m = lgb.LGBMClassifier(
        n_estimators=400, learning_rate=0.05, num_leaves=31,
        min_data_in_leaf=10, verbose=-1,
    ).fit(X.iloc[tr], y.iloc[tr])
    oof[te] = m.predict_proba(X.iloc[te])[:, 1]

bt_df = data.assign(prob_up=oof).dropna(subset=["prob_up"]).copy()
print(f"backtesting on {len(bt_df)} out-of-fold rows")

summaries = []
for tau in [0.0, 0.05, 0.10, 0.15, 0.20]:
    res = run_backtest(bt_df, threshold=tau)
    summaries.append({"threshold": tau, **res.summary()})
summary_df = pd.DataFrame(summaries).set_index("threshold").round(3)
summary_df


In [ ]:
# Equity curve: pick the threshold that gives the best Sharpe in this run and plot it.
best_tau = summary_df["annualised_sharpe"].idxmax()
best_res = run_backtest(bt_df, threshold=float(best_tau))
eq = best_res.equity_curve.copy()

fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(eq["date"], eq["strategy_equity"], label=f"L/S strategy (τ={best_tau})", linewidth=1.5)
ax.plot(eq["date"], eq["benchmark_equity"], label="SPY benchmark", linestyle="--", linewidth=1, color="grey")
ax.axhline(1.0, ls=":", c="black", alpha=0.4)
ax.set_title(f"Equity curve — out-of-fold predictions (τ={best_tau})\n"
             f"Sharpe {best_res.annualised_sharpe:.2f} | Max DD {best_res.max_drawdown:.2%} | "
             f"Hit-rate {best_res.hit_rate:.1%} | N={best_res.n_trades} trades")
ax.set_xlabel("Date"); ax.set_ylabel("Cumulative return (start=1)")
ax.grid(alpha=0.3); ax.legend()
plt.tight_layout(); plt.show()

print("\nTakeaway: the long/short rule beats SPY on this synthetic corpus. On REAL data the edge")
print("will be smaller and the walk-forward window more important — see README for how to run it.")


## 8. Productionisation — FastAPI + Streamlit

The entire inference path lives in `eca.api.main`. The relevant endpoints for a recruiter to know about:

```
POST /analyze
    Body: { ticker, call_date, prepared_remarks, qa_section }
    Returns: features (22 numbers) + prediction { prob_up, direction, confidence }

GET /features/{ticker}
    Returns all computed features for a ticker from the cached parquet

GET /backtest/{ticker}?threshold=0.1
    Runs the L/S backtest restricted to one ticker and returns the equity curve

GET /healthz        → { "status": "ok" }
```

The Streamlit UI (`streamlit_app.py`) wraps the API and adds two tabs: an *Analyze a call* form where you paste a raw transcript and see the features + prediction in real time, and a *Backtest* tab with the equity curve chart.

### Running it end-to-end on real data

```powershell
# 1. build features on ~500 HuggingFace transcripts (~5 min, no GPU needed)
python -m eca.cli build-dataset --source hf --limit 500

# 2. train (walk-forward CV, logs to MLflow)
python -m eca.cli train

# 3. cache out-of-sample predictions
python -m eca.cli predict

# 4. serve the API (in one terminal)
uvicorn eca.api.main:app --reload

# 5. launch the dashboard (in a second terminal)
streamlit run streamlit_app.py
```

Once the API is up, the `/docs` endpoint (auto-generated by FastAPI) shows the full OpenAPI spec — useful for a live demo with a recruiter.


## 9. Honest limitations & next steps

No portfolio project is complete without an honest accounting of what it *doesn't* do well. Recruiters who have built production systems will ask about exactly these points.

### What this project does well
- **Reproducible pipeline** — every step (ingest → features → model → backtest) is a Python module with a stable interface, unit-tested, and hooked into CI.
- **Causality-safe evaluation** — walk-forward CV and out-of-fold predictions ensure the backtest doesn't see the future.
- **Data ethics** — uses only public EDGAR filings and licensed HuggingFace datasets; the Motley Fool scraper is opt-in and robots.txt-aware.
- **Excess-return labelling** — removes market-wide moves so the signal is genuinely idiosyncratic.

### Known weaknesses

| Weakness | Why it matters | Mitigation path |
| --- | --- | --- |
| **Coverage** | EDGAR transcripts cover ~30 % of Russell 3000; many issuers only file the press release | Add the HF corpus (3.5k calls) and partner with a data vendor for production |
| **Look-ahead in feature engineering** | The current FLS detector uses *sentence-level* heuristics, not a lag-aware parser; a transcript leaking a post-close analyst revision would poison the label | Add a strict timestamp gate: features computed only from the call text up to the official filing timestamp |
| **Single-factor model** | We trade on *all* high-confidence calls equally; in practice a company in a sector with high earnings dispersion needs a different confidence threshold | Add sector-conditional thresholds or a sector fixed-effect in the LightGBM |
| **No transaction costs** | The backtest assumes zero slippage and frictionless execution | Add a per-trade cost of 2-5 bps and re-run; the Sharpe will drop but the model should still beat random |
| **Synthetic data here** | This notebook demonstrates the *pipeline* on synthetic data; the real-data results (to be filled in) live in the README table | Run `eca train` on the HuggingFace corpus and paste the CV metrics into the README |

### Next steps
1. **Docker** — multi-stage image, pre-pull FinBERT weights at build time, `docker compose up` for API + Streamlit. (Planned for the other machine where I have Docker Desktop.)
2. **Sector features** — add GICS sector code as a model feature and a sector-level volatility regressor.
3. **Transformer fine-tuning** — replace the classification head of FinBERT with a binary direction label; the sentence-aggregation approach may miss long-range dependencies in the Q&A section.
4. **Real-time EDGAR webhook** — poll the EDGAR EFTS API for new 8-K filings and trigger the pipeline automatically within minutes of filing.


---

## Summary

This project shows:

1. **Research literacy** — I know *why* earnings-call language predicts returns (Loughran-McDonald, Hassan et al.), not just *how* to run a transformer.
2. **End-to-end engineering** — ingest → features → model → backtest → API, each as a tested, importable Python module.
3. **Statistical discipline** — walk-forward CV, out-of-fold backtesting, excess-return labelling.
4. **Production thinking** — FastAPI service, Streamlit UI, CI (GitHub Actions), `.env`-based config, MLflow tracking.
5. **Data ethics** — explicit ToS-aware sourcing choices, documented trade-offs.

To reproduce everything on real data, follow the five commands in §8. The placeholders in the README Results table will fill in automatically after `eca train && eca predict && eca backtest`.
